# PA2 stage 04: persona LoRA-DPO with TRL (Task 5)

`HuggingFaceTB/SmolLM2-360M-Instruct` + LoRA trained with TRL `DPOTrainer` on
`data/persona_lora/ramsay_style_preferences.json` (200 train / 20 held out). Then the
untuned and LoRA-DPO models answer the fixed 40-prompt persona style set.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --format="%h %s"

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "peft==0.21.2" "trl==1.15.0"
!python -m pip install -q -e . --no-deps

In [ ]:
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2"

In [ ]:
# Link the attached prepared-data dataset to data/prepared, wherever Kaggle mounted it.
import glob
hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert hits, "attach the pa2-alignment-data dataset"
DATA_DIR = os.path.dirname(hits[0])
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/prepared"):
    os.symlink(DATA_DIR, "data/prepared")
!ls -la data/prepared/
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
# Resume support: copy this stage's run folders back from a previous run's attached output.
import shutil
for run in ['lora_dpo', 'gen_persona']:
    local = os.path.join("outputs", run)
    if os.path.exists(local):
        continue
    found = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}", recursive=True))
    if found:
        print(f"restoring outputs/{run} from {found[-1]}")
        shutil.copytree(found[-1], local, symlinks=False)
    else:
        print(f"outputs/{run}: starting fresh")
!du -sh outputs/* 2>/dev/null || true

In [ ]:
!mkdir -p outputs/lora_dpo
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -u scripts/train_lora_dpo.py --config configs/lora_dpo.yaml 2>&1 | tee -a outputs/lora_dpo/train.log

In [ ]:
# Persona eval set through the untuned SmolLM2 (cuda:1) and the LoRA-DPO merge (cuda:0), in parallel.
common = ("--eval_set eval_sets/persona_style_eval_40.json --batch_size 20 --max_new_tokens 256 --chat_template")
jobs = [
    ("1", f"python -u scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --generator smollm2-360m-instruct "
          f"--out outputs/gen_persona/smollm2_base.json {common}"),
    ("0", f"python -u scripts/generate.py --model outputs/lora_dpo/merged --generator smollm2-360m-lora-dpo "
          f"--out outputs/gen_persona/smollm2_lora_dpo.json {common}"),
]
os.makedirs("outputs/gen_persona", exist_ok=True)
procs = [subprocess.Popen(cmd, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONPATH": "."})
         for gpu, cmd in jobs]
print("exit codes", [p.wait() for p in procs])

In [ ]:
!PYTHONPATH=. python scripts/plot_metrics.py outputs/lora_dpo
import json
base = {r["id"]: r for r in json.load(open("outputs/gen_persona/smollm2_base.json"))}
lora = {r["id"]: r for r in json.load(open("outputs/gen_persona/smollm2_lora_dpo.json"))}
for rid in list(base)[::5]:
    print("###", base[rid]["category"], "|", base[rid]["instruction"])
    print("BASE:", base[rid]["output"][:400])
    print("LORA:", lora[rid]["output"][:400], "\n")

In [ ]:
!cd outputs && zip -qr /kaggle/working/lora_dpo_run_outputs.zip lora_dpo gen_persona -x "lora_dpo/merged/*" "lora_dpo/trainer/*"
!du -sh outputs/lora_dpo/* /kaggle/working/lora_dpo_run_outputs.zip